In [47]:
import pickle
import os
import numpy as np
import pandas as pd

import sys
sys.path.append("..")
from cd_zoo.tools.scoring_tools import score
from robustness_experiments.plot import renames

In [2]:
def clip_and_normalize( X):
    """
    Clip values to 2 standard deviations and normalize for each method independently.
    X shape: (batch, method, var, var, lag) or (batch, method, var, var)
    Each method slice is normalized independently over all other dimensions.
    """
    
    X = np.array(X)
    n_methods = X.shape[1]
    X_normalized = np.zeros_like(X)

    # before we do anything, clip occasionnally very large vectors to not skew the training.
    # THis is occasionally happening for models that return coefficients.
    X = X.clip(-10, 10)
    # Process each method independently
    for i in range(n_methods):
        method_slice = X[:, i, ...]
        # Calculate mean and std over all dimensions for this method
        mean = np.mean(method_slice)
        std = np.std(method_slice)
        # Clip to 2 standard deviations
        method_clipped = np.clip(method_slice, mean - 2 * std, mean + 2 * std)
        print("Clipping to:", mean - 2 * std, mean + 2 * std)
        # Normalize the clipped range to [0, 1]
        min_val = np.min(method_clipped)
        max_val = np.max(method_clipped)
        # Avoid division by zero
        range_val = max_val - min_val
        if range_val == 0:
            print("CAREFUL DIVISION BY ZERO IN NORMALIZATION, CHECK YOUR DATA!")
            range_val = 1
        X_normalized[:, i, ...] = (method_clipped - min_val) / range_val
    print(X_normalized.min(), X_normalized.max())
    return X_normalized

In [3]:
a = "/home/datasets4/stein/robust_exp/release_ensemble_results/test_corrected/small/WCG_wcg_results.p"

In [4]:
normal_formatting = pickle.load(open(a,"rb"))

m_order = normal_formatting[-1]  + ["varlingam","dynotears","fpcmci"]

In [5]:
stack = []
for method in m_order:
    p = f"/home/datasets4/stein/robust_exp/release_ensemble_rivers/causal_rivers_method_predictions/rivers/{method}/best_run/preds.p"
    b = pickle.load(open(p,"rb"))
    stack.append(b)

In [6]:
stack = np.stack(stack,axis=1)

In [7]:
stack = clip_and_normalize(stack)

Clipping to: 0.1476734657457327 1.2436312962316167
Clipping to: -0.40698073571462834 0.6251685808946749
Clipping to: 0.03783181154881854 1.346838671444369
Clipping to: 0.10704855271193636 1.2924781443690487
Clipping to: 0.12630994881107438 1.2728846829204155
Clipping to: -3.392945910275742 5.248005109783078
Clipping to: -1.9501532789332385 2.6570856186611342
Clipping to: -0.46216072424142407 0.628512297556733
Clipping to: -0.559324567370582 0.8523754070542697
0.0 1.0


In [8]:
pickle.dump(stack,open("/home/datasets4/stein/robust_exp/release_ensemble_rivers/processed_predictions.p","wb"))

# Calculate performance: 

In [9]:
_,Y  = pickle.load(open("/home/datasets4/stein/robust_exp/release_ensemble_rivers/releease_random_5_tcd_arena.p","rb"))
Y = np.stack(Y[:500])

In [56]:
# individual results:
individual = pd.read_csv("/home/datasets4/stein/robust_exp/release_ensemble_rivers/extracted/SG_max/rivers.csv", index_col=0)

In [13]:
individual["SHD individual"]

/home/datasets4/stein/robust_exp/release_ensemble_rivers/causal_rivers_method_predictions/rivers/cp//best_run/                  0.8508
/home/datasets4/stein/robust_exp/release_ensemble_rivers/causal_rivers_method_predictions/rivers/pcmci//best_run/               0.9777
/home/datasets4/stein/robust_exp/release_ensemble_rivers/causal_rivers_method_predictions/rivers/ntsnotears//best_run/          0.7583
/home/datasets4/stein/robust_exp/release_ensemble_rivers/causal_rivers_method_predictions/rivers/pcmciplus//best_run/           0.8353
/home/datasets4/stein/robust_exp/release_ensemble_rivers/causal_rivers_method_predictions/rivers/varlingam//best_run/           0.7745
/home/datasets4/stein/robust_exp/release_ensemble_rivers/causal_rivers_method_predictions/rivers/dynotears//best_run/           0.7146
/home/datasets4/stein/robust_exp/release_ensemble_rivers/causal_rivers_method_predictions/rivers/fpcmci//best_run/              0.8397
/home/datasets4/stein/robust_exp/release_ensemble_river

# Mean: 

In [14]:
preds = pickle.load(open("/home/datasets4/stein/robust_exp/release_ensemble_rivers/processed_predictions.p","rb"))

In [19]:
%%capture
mean = score(
    Y,
    preds.mean(axis=1).max(axis=-1),
    instant_labs=None,
    instant_preds=None,
    per_sample_metrics=True,
    remove_autoregressive_for_lagged=True
).loc["SHD individual"]

In [22]:
mean

SG_max    0.6592
Name: SHD individual, dtype: float64

# Ensembles

In [30]:
linear="/home/datasets4/stein/robust_exp/release_rivers_ensemble_performance/SimpleLinear/wcg_wcg/small/predictions_2026-02-28_07-46-09-775607.pkl"
mlp="/home/datasets4/stein/robust_exp/release_rivers_ensemble_performance/SimpleMLP/wcg_wcg/small/predictions_2026-03-01_17-59-15-747814.pkl"
transformer="/home/datasets4/stein/robust_exp/release_rivers_ensemble_performance/SimpleLinear/wcg_wcg/small/predictions_2026-02-28_07-46-09-775607.pkl"



linear = pickle.load(open(linear,"rb"))
mlp = pickle.load(open(mlp,"rb"))
transformer =  pickle.load(open(transformer,"rb"))

#maximum of last dimension:
linear = linear.max(axis=-1)
mlp = mlp.max(axis=-1)
transformer = transformer.max(axis=-1)


In [31]:
%%capture
linear_res = score(
    Y,
    linear,
    instant_labs=None,
    instant_preds=None,
    per_sample_metrics=True,
    remove_autoregressive_for_lagged=True

).loc["SHD individual"]

In [32]:
%%capture
mlp_res = score(
    Y,
    mlp,
    instant_labs=None,
    instant_preds=None,
    per_sample_metrics=True,
    remove_autoregressive_for_lagged=True

).loc["SHD individual"]

In [62]:
%%capture
transformer_res = score(
    Y,
    transformer,
    instant_labs=None,
    instant_preds=None,
    per_sample_metrics=True,
    remove_autoregressive_for_lagged=True

).loc["SHD individual"]

# Assemble: 

In [77]:
res_table = individual["SHD individual"]

res_table.index = [x.split("/")[-4] for x in res_table.index]
res_table.index  = [renames[x] for x in res_table.index]

In [78]:
res_table["Ensemble (Mean)"] = mean.values[0]
res_table["Ensemble (Linear)"] = linear_res.values[0]
res_table["Ensemble (MLP)"] = mlp_res.values[0]
res_table["Ensemble (Transformer)"] = transformer_res.values[0]

/tmp/ipykernel_1296040/1117565627.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_table["Ensemble (Mean)"] = mean.values[0]


In [79]:
print(pd.DataFrame(res_table).round(3).to_latex())

\begin{tabular}{lr}
\toprule
{} &  SHD individual \\
\midrule
CausalPretraining      &           0.851 \\
PCMCI                  &           0.978 \\
Nts-Notears            &           0.758 \\
PCMCI+                 &           0.835 \\
Varlingam              &           0.774 \\
Dynotears              &           0.715 \\
F-PCMCI                &           0.840 \\
SVAR-RFCI              &           0.857 \\
GVAR                   &           0.761 \\
CrossCorrelation       &           0.798 \\
Ensemble (Mean)        &           0.659 \\
Ensemble (Linear)      &           0.666 \\
Ensemble (MLP)         &           0.685 \\
Ensemble (Transformer) &           0.666 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_1296040/3529877854.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(pd.DataFrame(res_table).round(3).to_latex())
